# Huấn luyện NLU tiếng Việt cho chatbot TikTok Shop

Fine-tune **PhoBERT** (`vinai/phobert-base-v2`) cho hai việc:

1. **Phân loại ý định**: 7 nhãn `hoi_gia`, `tu_van`, `so_sanh`, `tra_don`, `bao_hanh`, `khieu_nai`, `khac`.
2. **Trích thông tin**: 7 loại thực thể (`SAN_PHAM`, `HANG`, `LOAI`, `NGAN_SACH`, `MAU`, `DUNG_LUONG`, `MA_DON`), gán nhãn BIO theo từ.

So với **baseline**: TF-IDF + Logistic Regression (ý định) và từ điển + biểu thức chính quy (thực thể).

## Cách chạy

**Colab**
1. Chép cả thư mục `nlu/` của repo lên Google Drive vào `MyDrive/cntttmdt/nlu` (sửa `NLU_DIR` ở cell cấu hình nếu để chỗ khác).
2. Mở notebook này trên Colab, chọn *Runtime > Change runtime type > T4 GPU*.
3. *Runtime > Run all*. Cho phép truy cập Drive khi được hỏi. Kết quả lưu vào `nlu/outputs/` trên Drive.

**Kaggle**
1. Nén thư mục `nlu/` thành zip, tạo Dataset mới trên Kaggle từ file zip đó.
2. Tạo Notebook, *File > Import Notebook* chọn file này; *Add Input* chọn dataset vừa tạo.
3. *Settings > Accelerator > GPU T4*, bật *Internet*. Chạy hết. Kết quả nằm ở tab *Output* (`/kaggle/working/outputs`).

Thời gian trên GPU T4: khoảng 10–15 phút cho cả hai mô hình.

> Con số trên tập **dev** chỉ để tham khảo: dev cũng là dữ liệu sinh tự động. Con số để đưa vào báo cáo là kết quả trên tập **test** do người viết (`nlu/data/test/*.jsonl`, xem `HUONG_DAN_GAN_NHAN.md`).

## 1. Cấu hình

In [ ]:
import os, sys, glob, json, time, random, subprocess, tempfile

SMOKE = os.environ.get("NLU_SMOKE") == "1"

MODEL_NAME = "vinai/phobert-base-v2"
EPOCHS_INTENT, LR_INTENT = 5, 3e-5
EPOCHS_NER, LR_NER = 8, 5e-5
BATCH = 32
MAX_LEN = 96
SEED = 42

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

if os.environ.get("NLU_DIR"):
    NLU_DIR = os.environ["NLU_DIR"]
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    NLU_DIR = "/content/drive/MyDrive/cntttmdt/nlu"
elif IN_KAGGLE:
    hits = glob.glob("/kaggle/input/**/nlu_lib.py", recursive=True)
    assert hits, "Chưa thêm dataset chứa thư mục nlu (Add Input)"
    NLU_DIR = os.path.dirname(hits[0])
else:
    NLU_DIR = os.path.abspath(".")

assert os.path.exists(os.path.join(NLU_DIR, "nlu_lib.py")), f"Không thấy nlu_lib.py trong {NLU_DIR}"
OUT_DIR = os.environ.get("NLU_OUT") or ("/kaggle/working/outputs" if IN_KAGGLE else os.path.join(NLU_DIR, "outputs"))
CKPT_DIR = os.path.join(tempfile.gettempdir(), "nlu_ckpt") if (IN_COLAB or IN_KAGGLE) else os.path.join(OUT_DIR, "ckpt")
os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, NLU_DIR)

print("Môi trường:", "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "máy cá nhân", "| CHẠY THỬ NHỎ" if SMOKE else "")
print("Thư mục nlu:", NLU_DIR)
print("Lưu kết quả:", OUT_DIR)

## 2. Cài thư viện
`underthesea` để tách từ tiếng Việt (PhoBERT cần câu đã tách từ), `seqeval` để chấm thực thể.

In [ ]:
if IN_COLAB or IN_KAGGLE:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "underthesea", "seqeval", "accelerate"])

import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline, make_union
from seqeval.metrics import classification_report as seq_report
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_p, recall_score as seq_r

import nlu_lib as L
import templates_vi as T

transformers.set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| thiết bị:", DEVICE,
      torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
if DEVICE == "cpu" and not SMOKE:
    print("CẢNH BÁO: chưa bật GPU, train sẽ rất chậm. Colab: Runtime > Change runtime type > T4 GPU.")
assert L._get_segmenter(), "underthesea chưa cài được"

## 3. Đọc dữ liệu
Train = câu sinh từ mẫu (+ câu Gemini viết lại, + câu người viết trong `data/human/` nếu có). Dev = mẫu câu và sản phẩm không có trong train. Test = câu người viết trong `data/test/`.

In [ ]:
DATA = os.path.join(NLU_DIR, "data")
gen_dir = os.path.join(DATA, "generated")
if not os.path.exists(os.path.join(gen_dir, "train.jsonl")):
    catalog = os.path.join(DATA, "catalog.csv")
    assert os.path.exists(catalog), (
        f"Thiếu {catalog}. Thư mục nlu trên Drive chưa có đủ data/: tải cả thư mục nlu/data "
        "từ repo lên MyDrive/cntttmdt/nlu/ (đè bản cũ), rồi chạy lại.")
    print("Không thấy data/generated/train.jsonl, sinh lại dữ liệu từ mẫu câu...")
    gen_dir = os.path.join(OUT_DIR, "generated")
    r = subprocess.run([sys.executable, os.path.join(NLU_DIR, "gen_data.py"), "--out", gen_dir],
                       capture_output=True, text=True, encoding="utf-8",
                       env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    print(r.stdout[-1500:])
    if r.returncode != 0:
        raise RuntimeError("gen_data.py lỗi:" + chr(10) + r.stderr[-3000:])

train = L.read_jsonl(os.path.join(gen_dir, "train.jsonl"))
for extra in [os.path.join(gen_dir, "gemini.jsonl"), *sorted(glob.glob(os.path.join(DATA, "human", "*.jsonl")))]:
    if os.path.exists(extra):
        rows = L.read_jsonl(extra)
        train += rows
        print(f"+ {len(rows)} câu từ {os.path.basename(extra)}")
dev = L.read_jsonl(os.path.join(gen_dir, "dev.jsonl"))
test = [r for f in sorted(glob.glob(os.path.join(DATA, "test", "*.jsonl"))) for r in L.read_jsonl(f)]
meta = json.load(open(os.path.join(gen_dir, "meta.json"), encoding="utf-8"))

if SMOKE:
    random.Random(0).shuffle(train)
    train, dev = train[:70], dev[::16]
    test = test[:20] or dev[:10]

if not test:
    print("CHƯA CÓ TẬP TEST do người viết (data/test/*.jsonl): chỉ đánh giá trên dev.")
SPLITS = {"dev": dev, **({"test": test} if test else {})}
print({k: len(v) for k, v in {"train": train, **SPLITS}.items()})
pd.Series([r["intent"] for r in train]).value_counts()

## 4. Tách từ và gán nhãn BIO
Ví dụ `chuột logitech g102 còn màu trắng` → cụm `chuột logitech g102` được gán `B-SAN_PHAM I-SAN_PHAM I-SAN_PHAM`. Từ ghép tiếng Việt nối bằng `_` (vd `bao_nhiêu`).

In [ ]:
t0 = time.time()
for rows in (train, *SPLITS.values()):
    for r in rows:
        r["seg"] = L.segmented_text(r["text"])
        r["tokens"], r["bio"], r["spans"] = L.to_bio(r["text"], r["entities"])
print(f"Tách từ {sum(len(x) for x in (train, *SPLITS.values()))} câu trong {time.time() - t0:.1f}s")

ex = next(r for r in train if len(r["entities"]) >= 2)
print(ex["text"])
print(ex["seg"])
print(list(zip(ex["tokens"], ex["bio"])))

## 5. Baseline
- **Ý định:** TF-IDF (từ 1–2 gram + ký tự 2–5 gram) trên câu đã bỏ dấu, chữ thường, rồi Logistic Regression.
- **Thực thể:** tra từ điển tên sản phẩm, hãng, loại, màu (chỉ dùng sản phẩm có trong train) + biểu thức chính quy cho số tiền, dung lượng, mã đơn.

In [ ]:
baseline_intent = make_pipeline(
    make_union(TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True),
               TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True)),
    LogisticRegression(max_iter=3000, C=10),
)
baseline_intent.fit([L.fold(r["text"]) for r in train], [r["intent"] for r in train])

catalog = L.load_catalog(os.path.join(DATA, "catalog.csv"))
train_skus = set(meta["train_products"])
gazetteer = L.GazetteerNER(
    [p for p in catalog if p.sku in train_skus],
    extra_categories=T.EXTRA_CATEGORIES + T.AUDIO_CATEGORIES,
    extra_colors=T.EXTRA_COLORS,
    extra_brands=T.EXTRA_BRANDS + T.PHONE_BRANDS,
    extra_products=T.DEVICES,
)

pred = {}
for split, rows in SPLITS.items():
    pred[split] = {"Baseline": (
        list(baseline_intent.predict([L.fold(r["text"]) for r in rows])),
        [L.spans_to_labels(r["spans"], gazetteer.predict(r["text"])) for r in rows],
    )}
print("Xong baseline")

## 6. PhoBERT: phân loại ý định

In [ ]:
from transformers import (AutoModelForSequenceClassification, AutoModelForTokenClassification, AutoTokenizer,
                          DataCollatorForTokenClassification, DataCollatorWithPadding, Trainer, TrainingArguments)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class ListDataset(torch.utils.data.Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


def train_args(name, epochs, lr):
    steps_per_epoch = max(1, len(train) // BATCH)
    kw = dict(
        output_dir=os.path.join(CKPT_DIR, name),
        learning_rate=lr,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=64,
        num_train_epochs=epochs,
        weight_decay=0.01,
        warmup_steps=int(0.1 * steps_per_epoch * epochs),
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=False,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=1,
        logging_steps=20,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED,
    )
    if SMOKE:
        kw.update(max_steps=2, eval_strategy="steps", save_strategy="steps", eval_steps=2, save_steps=2,
                  logging_steps=1, warmup_steps=0)
    return TrainingArguments(**kw)


def reload_best(trainer, model_cls):
    best = trainer.state.best_model_checkpoint
    if best:
        print(f"Dùng checkpoint tốt nhất: {os.path.basename(best)} (F1 dev = {trainer.state.best_metric:.4f})")
        trainer.model = model_cls.from_pretrained(best).to(trainer.args.device)
    return trainer.model


intent2id = {k: i for i, k in enumerate(L.INTENTS)}
id2intent = {i: k for k, i in intent2id.items()}


def encode_intent(rows):
    return ListDataset([{**tokenizer(r["seg"], truncation=True, max_length=MAX_LEN), "labels": intent2id[r["intent"]]}
                        for r in rows])


def metrics_intent(p):
    y = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, y), "f1": f1_score(p.label_ids, y, average="macro")}


model_intent = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(L.INTENTS), id2label=id2intent, label2id=intent2id)
trainer_intent = Trainer(
    model=model_intent,
    args=train_args("intent", EPOCHS_INTENT, LR_INTENT),
    train_dataset=encode_intent(train),
    eval_dataset=encode_intent(dev),
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=metrics_intent,
)
t0 = time.time()
trainer_intent.train()
TRAIN_TIME = {"intent_s": round(time.time() - t0, 1)}
reload_best(trainer_intent, AutoModelForSequenceClassification)
print(f"Train ý định xong trong {TRAIN_TIME['intent_s']}s")

## 7. PhoBERT: trích thông tin (NER)
Tokenizer của PhoBERT không cho biết subword thuộc từ nào, nên `nlu_lib.encode_ner_example` tự gióng: nhãn đặt ở subword đầu của mỗi từ, các subword còn lại bỏ qua khi tính loss.

In [ ]:
label2id = {l: i for i, l in enumerate(L.BIO_LABELS)}
id2label = {i: l for l, i in label2id.items()}


def encode_ner(rows):
    items = []
    for r in rows:
        e = L.encode_ner_example(tokenizer, r["tokens"], r["bio"], label2id, MAX_LEN)
        e.pop("word_index")
        items.append(e)
    return ListDataset(items)


def metrics_ner(p):
    y = p.predictions.argmax(-1)
    gold, guess = [], []
    for pr, lb in zip(y, p.label_ids):
        gold.append([id2label[l] for l in lb if l != -100])
        guess.append([id2label[x] for x, l in zip(pr, lb) if l != -100])
    return {"precision": seq_p(gold, guess, zero_division=0), "recall": seq_r(gold, guess, zero_division=0),
            "f1": seq_f1(gold, guess, zero_division=0)}


model_ner = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(L.BIO_LABELS), id2label=id2label, label2id=label2id)
trainer_ner = Trainer(
    model=model_ner,
    args=train_args("ner", EPOCHS_NER, LR_NER),
    train_dataset=encode_ner(train),
    eval_dataset=encode_ner(dev),
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=metrics_ner,
)
t0 = time.time()
trainer_ner.train()
TRAIN_TIME["ner_s"] = round(time.time() - t0, 1)
reload_best(trainer_ner, AutoModelForTokenClassification)
print(f"Train NER xong trong {TRAIN_TIME['ner_s']}s")

## 8. Hàm dự đoán
Đây cũng là logic mà service FastAPI sẽ dùng khi chạy thật.

In [ ]:
@torch.no_grad()
def predict_intent(texts, model=None, batch=64):
    model = model or trainer_intent.model
    model.eval()
    segs = [L.segmented_text(t) for t in texts]
    out = []
    for i in range(0, len(segs), batch):
        enc = tokenizer(segs[i:i + batch], truncation=True, max_length=MAX_LEN, padding=True,
                        return_tensors="pt").to(model.device)
        probs = model(**enc).logits.softmax(-1).cpu()
        out += [(id2intent[int(p.argmax())], round(float(p.max()), 4)) for p in probs]
    return out


@torch.no_grad()
def predict_word_labels(tokens, model=None):
    model = model or trainer_ner.model
    model.eval()
    enc = L.encode_ner_example(tokenizer, tokens, None, label2id, MAX_LEN)
    ids = torch.tensor([enc["input_ids"]], device=model.device)
    best = model(input_ids=ids, attention_mask=torch.ones_like(ids)).logits[0].argmax(-1).cpu().tolist()
    labels = ["O"] * len(tokens)
    for pos, wi in enumerate(enc["word_index"]):
        if wi >= 0:
            labels[wi] = id2label[best[pos]]
    return labels


def predict_entities(text, model=None):
    text = L.normalize(text)
    tokens, _, spans = L.to_bio(text, [])
    ents = L.bio_to_entities(text, spans, predict_word_labels(tokens, model))
    for e in ents:
        if e["label"] == "NGAN_SACH":
            e["value"] = L.parse_money(e["text"])
    return ents


def analyze(text):
    (intent, conf), = predict_intent([text])
    return {"text": text, "intent": intent, "confidence": conf, "entities": predict_entities(text)}


for split, rows in SPLITS.items():
    pred[split]["PhoBERT"] = (
        [p for p, _ in predict_intent([r["text"] for r in rows])],
        [predict_word_labels(r["tokens"]) for r in rows],
    )
analyze("sac anker 20w con mau trang ko shop, tam 500k co loai nao ko")

## 9. So sánh với baseline
`khong_dau`: chỉ tính trên các câu gõ không dấu, để xem mô hình có chịu được kiểu nhắn nhanh không.

In [ ]:
def is_no_diacritics(r):
    t = r["text"]
    return t == L.strip_diacritics(t) and any(c.isalpha() for c in t)


def score(rows, intents, bio):
    gold_i = [r["intent"] for r in rows]
    gold_n = [r["bio"] for r in rows]
    nd = [i for i, r in enumerate(rows) if is_no_diacritics(r)]
    res = {
        "intent_accuracy": accuracy_score(gold_i, intents),
        "intent_macro_f1": f1_score(gold_i, intents, average="macro", labels=L.INTENTS, zero_division=0),
        "ner_precision": seq_p(gold_n, bio, zero_division=0),
        "ner_recall": seq_r(gold_n, bio, zero_division=0),
        "ner_f1": seq_f1(gold_n, bio, zero_division=0),
        "so_cau_khong_dau": len(nd),
    }
    if nd:
        res["intent_macro_f1_khong_dau"] = f1_score([gold_i[i] for i in nd], [intents[i] for i in nd],
                                                    average="macro", zero_division=0)
        res["ner_f1_khong_dau"] = seq_f1([gold_n[i] for i in nd], [bio[i] for i in nd], zero_division=0)
    return res


RESULTS = {f"{split}/{model}": score(SPLITS[split], *pred[split][model]) for split in SPLITS for model in pred[split]}
table = pd.DataFrame(RESULTS).T
table.style.format("{:.3f}", subset=[c for c in table.columns if c != "so_cau_khong_dau"])

In [ ]:
FINAL = "test" if "test" in SPLITS else "dev"
rows = SPLITS[FINAL]
for model in ("Baseline", "PhoBERT"):
    intents, bio = pred[FINAL][model]
    print(f"===== {model} trên {FINAL}: ý định")
    print(classification_report([r["intent"] for r in rows], intents, labels=L.INTENTS, zero_division=0, digits=3))
    print(f"===== {model} trên {FINAL}: thực thể")
    print(seq_report([r["bio"] for r in rows], bio, zero_division=0, digits=3))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, model in zip(axes, ("Baseline", "PhoBERT")):
    cm = confusion_matrix([r["intent"] for r in rows], pred[FINAL][model][0], labels=L.INTENTS)
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(L.INTENTS)), L.INTENTS, rotation=45, ha="right")
    ax.set_yticks(range(len(L.INTENTS)), L.INTENTS)
    for i in range(len(L.INTENTS)):
        for j in range(len(L.INTENTS)):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xlabel("Dự đoán")
    ax.set_ylabel("Nhãn đúng")
    ax.set_title(f"{model} – ý định ({FINAL})")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f"ma_tran_nham_lan_{FINAL}.png"), dpi=150)
plt.show()

## 10. Phân tích lỗi
Ghi các câu đoán sai ra `loi_y_dinh.csv` và `loi_thuc_the.csv` để đọc và viết vào báo cáo.

In [ ]:
err_i, err_n = [], []
for split, rows in SPLITS.items():
    pi, pn = pred[split]["PhoBERT"]
    bi, bn = pred[split]["Baseline"]
    for r, p, b, lab, blab in zip(rows, pi, bi, pn, bn):
        if p != r["intent"]:
            err_i.append({"split": split, "text": r["text"], "dung": r["intent"], "phobert": p, "baseline": b})
        if lab != r["bio"]:
            guess = L.bio_to_entities(r["text"], r["spans"], lab)
            err_n.append({"split": split, "dung": L.to_markup(r["text"], r["entities"]),
                          "phobert": L.to_markup(r["text"], guess)})
pd.DataFrame(err_i).to_csv(os.path.join(OUT_DIR, "loi_y_dinh.csv"), index=False, encoding="utf-8-sig")
pd.DataFrame(err_n).to_csv(os.path.join(OUT_DIR, "loi_thuc_the.csv"), index=False, encoding="utf-8-sig")
print(f"{len(err_i)} câu sai ý định, {len(err_n)} câu sai thực thể (PhoBERT)")
pd.DataFrame(err_n).head(15)

## 11. Tốc độ
Thời gian xử lý một tin nhắn (tách từ + 2 mô hình). Service chạy thật trên laptop sẽ dùng CPU.

In [ ]:
sample = [r["text"] for r in dev[:50]]


def bench(device):
    m_i, m_n = trainer_intent.model.to(device), trainer_ner.model.to(device)
    for t in sample[:3]:
        predict_intent([t], m_i), predict_entities(t, m_n)
    t0 = time.perf_counter()
    for t in sample:
        predict_intent([t], m_i)
        predict_entities(t, m_n)
    return round((time.perf_counter() - t0) / len(sample) * 1000, 1)


LATENCY = {"cpu_ms_moi_cau": bench("cpu")}
if torch.cuda.is_available():
    LATENCY["gpu_ms_moi_cau"] = bench("cuda")
trainer_intent.model.to(DEVICE)
trainer_ner.model.to(DEVICE)
LATENCY

## 12. Lưu mô hình và kết quả
- `intent_model/`, `ner_model/`: mô hình đã train (dùng cho service FastAPI)
- `metrics.json`, `so_sanh.csv`: bảng so sánh để đưa vào báo cáo

In [ ]:
for name, tr in (("intent_model", trainer_intent), ("ner_model", trainer_ner)):
    path = os.path.join(OUT_DIR, name)
    tr.model.save_pretrained(path)
    tokenizer.save_pretrained(path)

summary = {
    "model": MODEL_NAME,
    "smoke": SMOKE,
    "time": time.strftime("%Y-%m-%d %H:%M"),
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
    "data": {"train": len(train), **{k: len(v) for k, v in SPLITS.items()}, "generated_meta": meta["counts"]},
    "hyperparams": {"epochs_intent": EPOCHS_INTENT, "lr_intent": LR_INTENT, "epochs_ner": EPOCHS_NER,
                    "lr_ner": LR_NER, "batch": BATCH, "max_len": MAX_LEN, "seed": SEED},
    "train_time_s": TRAIN_TIME,
    "results": RESULTS,
    "latency": LATENCY,
}
with open(os.path.join(OUT_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
table.to_csv(os.path.join(OUT_DIR, "so_sanh.csv"), encoding="utf-8-sig")
print("Đã lưu vào", OUT_DIR)
print(sorted(os.listdir(OUT_DIR)))

## 13. Thử vài câu

In [ ]:
for t in [
    "chuot g102 con mau trang ko shop",
    "tầm 15tr nên mua laptop nào để học ạ",
    "đơn 579123456789012345 tới đâu rồi shop",
    "tai nghe sony mới nhận bị rè một bên, chán thật",
    "samsung 870 evo với kingston nv2 cái nào nhanh hơn",
    "cảm ơn shop nha",
]:
    a = analyze(t)
    ents = ", ".join(f"{e['label']}={e['text']}" + (f" ({e['value']:,}đ)" if e.get("value") else "") for e in a["entities"])
    print(f"{a['intent']:10s} {a['confidence']:.2f} | {t}\n{'':16s}{ents}")